## Импорт библиотек и загрузка данных

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
import pandas as pd
import sqlite3
import os



# Создаем подключение к базе
conn = sqlite3.connect('edtech_data.db')


pd.read_csv('users.csv').to_sql('users', conn, if_exists='replace', index=False)
pd.read_csv('payments.csv').to_sql('payments', conn, if_exists='replace', index=False)
pd.read_csv('trial_lessons.csv').to_sql('trial_lessons', conn, if_exists='replace', index=False)
pd.read_csv('ad_spend.csv').to_sql('ad_spend', conn, if_exists='replace', index=False)
pd.read_csv('funnel_events.csv').to_sql('funnel_events', conn, if_exists='replace', index=False)
pd.read_csv('test_results.csv').to_sql('test_results', conn, if_exists='replace', index=False)

print("Все таблицы загружены!")

# Подготовка данных 

## Проверка количества строк в каждой таблице

Вывод: Данные успешно загружены. В таблице users 1500 пользователей, в funnel_events 5570 событий, в payments 163 платежа. Объёма данных достаточно для первичного анализа пользовательской воронки, каналов и оплат.

In [ ]:
query = """SELECT COUNT(*) AS strings_users
,(SELECT COUNT(*) FROM funnel_events ) AS strings_funnel_events
,(SELECT COUNT(*) FROM test_results) AS strings_test_results
,(SELECT COUNT(*) FROM trial_lessons) AS strings_trial_lessons
,(SELECT COUNT(*) FROM payments) AS strings_payments
,(SELECT COUNT(*) FROM ad_spend ) AS strings_ad_spend
 FROM users """
pd.read_sql(query, conn)

## Уникальность user_id в таблице users

Вывод: Количество уникальных user_id совпадает с общим количеством строк в таблице users. Это значит, что в таблице нет дубликатов пользователей, и её можно использовать как базовую таблицу для анализа воронки и каналов.

In [ ]:
query = """
SELECT 
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(*) AS total_rows
FROM users;

"""
pd.read_sql(query, conn)

## Есть ли пользователи в событиях, которых нет в users

Вывод: Аномальных ID не обнаружено(Перед проблемных мест в пути пользователя по группам,нужно убедиться, что мы работаем с одинаковыми пользователями и данные собраны и интерпретированы правильно)

In [ ]:
query = "SELECT DISTINCT user_id FROM  funnel_events WHERE user_id not in(SELECT DISTINCT user_id FROM users) "
pd.read_sql(query, conn)

## Пропуски в ключевых полях

Вывод:Критичных пропусков в ключевых полях не обнаружено. Пропуски в test_completed_at, test_score, recommended_level и recommended_plan связаны с тем, что часть пользователей начала тест, но не завершила его. Пропуски в trial_attended_at и teacher_rating связаны с тем, что часть пользователей записалась на пробный урок, но не посетила его. Такие пропуски являются частью пользовательского поведения и могут быть использованы в дальнейшем анализе отвалов.

### Пропуски user_id

In [ ]:
query = """SELECT COUNT(*) AS id_null_users
 ,(SELECT COUNT(*) FROM funnel_events WHERE user_id IS NULL ) AS id_null_funnel_events
 ,(SELECT COUNT(*) FROM payments WHERE user_id IS NULL) AS id_null_payments
 ,(SELECT COUNT(*) FROM trial_lessons WHERE user_id IS NULL) AS id_null_trial_lessons
 ,(SELECT COUNT(*) FROM test_results WHERE user_id IS NULL) AS id_null_test_results
 FROM users WHERE user_id IS NULL
 """
pd.read_sql(query, conn)

### Пропуски event_name и event_time

In [ ]:
query = "SELECT COUNT(*) FROM funnel_events WHERE event_name IS NULL"
pd.read_sql(query, conn)

In [ ]:
query = "SELECT COUNT(*) FROM funnel_events WHERE event_time IS NULL"
pd.read_sql(query, conn) 

### Пропуски acquisition_channel

In [ ]:
query = """
SELECT COUNT(*) AS channel_null_funnel_events
,(SELECT COUNT(*) FROM payments WHERE acquisition_channel IS NULL) AS channel_null_payments
,(SELECT COUNT(*) FROM ad_spend WHERE acquisition_channel IS NULL) AS channel_null_ad_spend
,(SELECT COUNT(*) FROM users WHERE acquisition_channel IS NULL) AS channel_null_users
FROM funnel_events WHERE acquisition_channel IS NULL
"""
pd.read_sql(query, conn) 

### Пропуски payment_status and amount_eur 

In [ ]:
query = "SELECT COUNT(*) FROM payments WHERE payment_status IS NULL"
pd.read_sql(query, conn)

In [ ]:
query = "SELECT COUNT(*) FROM payments WHERE amount_eur  IS NULL"
pd.read_sql(query, conn)

### Провека всех строк

In [ ]:
trial_lessons = pd.read_sql("SELECT * FROM trial_lessons", conn)
print(trial_lessons.isnull().sum())
print()
test_results = pd.read_sql("SELECT * FROM test_results", conn)
print(test_results.isnull().sum())
print()
users = pd.read_sql("SELECT * FROM users", conn)
print(users.isnull().sum())
print()
ad_spend = pd.read_sql("SELECT * FROM ad_spend", conn)
print(ad_spend.isnull().sum())
print()
funnel_events = pd.read_sql("SELECT * FROM funnel_events", conn)
print(funnel_events.isnull().sum())
print()
payments = pd.read_sql("SELECT * FROM payments", conn)
print(payments.isnull().sum())

## Дубликаты событий

Вывод:Дубликаты по event_id не обнаружены. Также не выявлены повторяющиеся события по комбинации user_id, event_time и event_name.

In [ ]:
query = """SELECT 
    COUNT(DISTINCT event_id) AS unique_event_ids,
    COUNT(*) AS total_events
FROM funnel_events

 """
pd.read_sql(query, conn)

In [ ]:
query = """SELECT 
    user_id,
    event_time,
    event_name,
    COUNT(*) AS cnt
FROM funnel_events
GROUP BY 
    user_id,
    event_time,
    event_name
HAVING COUNT(*) > 1

 """
pd.read_sql(query, conn)

## Корректность дат

Вывод: В таблице funnel_events не обнаружены случаи, когда события пользователя идут в обратном хронологическом порядке.

Не обнаружено случаев оплаты раньше первого визита

Не обнаружено посещений раньше записи

Даты коректны и готовы к работе

In [ ]:
query = """ 
WITH tb AS( SELECT user_id
,event_time AS t1
,LAG(event_time,1) OVER(PARTITION BY user_id ORDER BY event_time) AS t2
FROM funnel_events  )
SELECT user_id, t1 AS event_time FROM tb
 WHERE     t1 < t2         
"""
pd.read_sql(query, conn)

In [ ]:
query = """ 
SELECT 
    p.user_id,
    u.first_visit_at,
    p.payment_date
FROM payments p
JOIN users u 
    ON p.user_id = u.user_id
WHERE p.payment_date < DATE(u.first_visit_at     )   
"""
pd.read_sql(query, conn)

In [ ]:
query = """ 
SELECT 
    user_id,
    trial_booked_at,
    trial_attended_at
FROM trial_lessons
WHERE trial_attended_at IS NOT NULL
  AND trial_attended_at < trial_booked_at

"""

pd.read_sql(query, conn)

## Корректность Платежей

ВЫВОДЫ: 

Каждый payment_id встречается один раз, дубликатов платежей не обнаружено. По статусам видно, что часть платежей завершилась ошибкой, поэтому этот этап требует отдельного анализа.

В текущей таблице payments нет платежей со статусом refunded. Это не обязательно означает проблему с возвратами. Нужно уточнить, действительно ли возвратов не было за период анализа или данные по возвратам хранятся отдельно.

Доля неуспешных платежей составляет 22.7%. Это заметная доля, поэтому этап оплаты требует дополнительной проверки. Возможные причины: технические ошибки, неудобный платёжный сценарий, недостаток способов оплаты, ошибки со стороны пользователя или отсутствие повторной попытки после failed payment.

ГИПОТЕЗЫ:

Гипотеза 1. Пользователи не повторяют попытку оплаты после ошибки, поэтому часть потенциальной выручки теряется.

Гипотеза 2. На этапе оплаты может быть техническая или UX-проблема, но для подтверждения нужны дополнительные данные: payment_method, error_code, device, country и повторные попытки оплаты.

Гипотеза 3. Пользователям может не хватать удобных способов оплаты.

РЕКОМЕНДАЦИИ:

1.Проверить коректность работы возвратов

2.Проверить коректность работы терминала оплаты или сбора данных с него

3.В случае коректной работы всех систем, Для пользователей которые ушли с ошибкой, провести ретаргетинг с предложением о скидке + Email-рассылку


In [ ]:
query = """
SELECT 
    COUNT(DISTINCT payment_id) AS unique_payment_ids,
    COUNT(*) AS total_payments
FROM payments
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT 
    payment_status,
    COUNT(*) AS payments_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM payments), 1) AS share_percent
FROM payments
GROUP BY payment_status
ORDER BY payments_count DESC

"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT user_id 
,COUNT(payment_id) AS cnt_status
 FROM payments GROUP BY user_id   HAVING COUNT(payment_id) > 1

"""
pd.read_sql(query, conn)

# Базовый анализ пользователей

Выводы: 

* 1.Наибольшее количество пользователей приходит из TikTok(27.3%) и Instagram(21.7 %), которые суммарно обеспечивают почти половину всех новых регистраций

* 2.Анализ внутренних кампаний показывает разную степень однородности трафика. В Google Ads и YouTube наблюдается наиболее стабильное распределение (разброс до 0.7%), что говорит о предсказуемости этих инструментов. В то же время в TikTok и Instagram выделяются явные лидеры (кампании со скидками).

* 3.Наибольшее количество пользователей используют Телефоны(66.3%)

* 4.Основным рынком является Германия, откуда приходит подавляющее большинство пользователей(71.4%)

* 5.Ключевым стимулом для изучения языка является развитие карьеры (28.3%), что важно учитывать при подготовке рекламных офферов.(28.3 %) 

* 6.Основной точкой входа для новых пользователей является страница прохождения теста на уровень языка (41.5%)

## SQL Запросы

In [ ]:
query = """
SELECT acquisition_channel	
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_channel_percent
 FROM users GROUP BY acquisition_channel ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

In [ ]:


query = """
SELECT 
    acquisition_channel
    ,campaign
    ,COUNT(user_id) AS all_users_campaign
    ,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1) || ' %' AS users_campaign_percent
FROM users
GROUP BY acquisition_channel, campaign
ORDER BY SUM(COUNT(user_id)) OVER(PARTITION BY acquisition_channel) DESC, all_users_campaign DESC
"""
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT device
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_device_percent
 FROM users GROUP BY device ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT country	
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_country_percent
 FROM users GROUP BY country ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT user_goal	
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_goal_percent
 FROM users GROUP BY user_goal ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

In [ ]:
query = """
SELECT landing_page		
,COUNT( user_id) AS all_users
,ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM users), 1)|| ' %' AS users_landing_page_percent
 FROM users GROUP BY landing_page	 ORDER BY all_users DESC
 """
pd.read_sql(query, conn)

# Общая пользовательская воронка и Анализ отвалов в воронке 

Вывод: 
* Максимальное число пользователей уходит на этапе начала теста (514 человек) и бронирования пробного занятия (392 человека). 

*  Критически низкая конверсия наблюдается при переходе к бронированию пробного занятия (отвал 52.1%) и на этапе оплаты обучения (отвал 55.3%). Более половины аудитории, дошедшей до этих шагов, покидает воронку

* Отвал после первого визита может  указывать на несоответствие рекламного оффера содержимому страницы, проблемы с UX/UI (сложная навигация, долгая загрузка) в ожидании пользователя,недостаточную ценность бесплатного теста для пользователя.

* Отвал после регистрации может означать не подходящие варианты записи на пробный урок,не интересные форматы проведения,нарушение ожиданий

* Отвал на этапе оплаты может означать проблемы в качестве пробного урока,скрипте продаж,несовпадение с ценовыми ожиданиями,недостаток вариантов оплаты(подписка,рассрочка,единоразовый платёж)

In [ ]:
query = """
WITH t1 AS (
    SELECT 
        event_name,
        COUNT(DISTINCT user_id) AS users_count,
        CASE 
            WHEN event_name = 'site_visit' THEN 1
            WHEN event_name = 'test_started' THEN 2
            WHEN event_name = 'test_completed' THEN 3
            WHEN event_name = 'registration' THEN 4
            WHEN event_name = 'trial_booked' THEN 5
            WHEN event_name = 'trial_attended' THEN 6
            WHEN event_name = 'plan_selected' THEN 7
            WHEN event_name = 'payment_success' THEN 8
        END AS step_number
    FROM funnel_events
    WHERE event_name IN ('site_visit', 'test_started', 'test_completed', 
                         'registration', 'trial_booked', 'trial_attended', 
                         'plan_selected', 'payment_success')
    GROUP BY event_name)

SELECT
    event_name
    ,users_count
    ,ROUND(users_count * 100.0 / (SELECT MAX(users_count) FROM t1), 1)|| ' %' AS total_conv
    ,ROUND(users_count * 100.0 / LAG(users_count) OVER (ORDER BY users_count DESC), 1) || ' %' AS step_conv
    , LAG(users_count) OVER (ORDER BY users_count DESC) - users_count   AS users_churn
    , ROUND((LAG(users_count) OVER (ORDER BY users_count DESC) - users_count) * 100.0 / 
      LAG(users_count) OVER (ORDER BY users_count DESC), 1) || ' %' AS step_churn_percent
FROM t1
GROUP BY event_name ORDER BY step_number
       """
pd.read_sql(query, conn)